# SUPG objectives and direct optimization

**Status:** canonical foundation notebook  
**Thesis sections:** 1.1–1.2

This notebook keeps the *mathematical* FEM setup visible: mesh, function spaces, boundary data, weak form, SUPG term, and the objective. `supgml` takes over only repetitive assembly, sparse-solver lifecycle, and optimization plumbing.

Run it in the DOLFINx kernel after installing this repository with `%pip install -e "..[ml,viz]"`.

## 1. Define a singularly perturbed convection–diffusion problem

We solve $-\varepsilon\Delta u + \boldsymbol b\cdot\nabla u=f$ on the unit square with homogeneous Dirichlet data. The small diffusion coefficient makes the problem convection dominated.

In [ ]:
import numpy as np
import ufl
from dolfinx import default_scalar_type, fem, mesh as dmesh
from mpi4py import MPI

mesh = dmesh.create_unit_square(MPI.COMM_WORLD, 24, 24)
V = fem.functionspace(mesh, ("CG", 1))
u_h = fem.Function(V, name="state")
u, v = ufl.TrialFunction(V), ufl.TestFunction(V)

mesh.topology.create_connectivity(mesh.topology.dim - 1, mesh.topology.dim)
facets = dmesh.exterior_facet_indices(mesh.topology)
dofs = fem.locate_dofs_topological(V, mesh.topology.dim - 1, facets)
bc = fem.dirichletbc(fem.Constant(mesh, default_scalar_type(0.0)), dofs, V)

eps = fem.Constant(mesh, default_scalar_type(1e-3))
b = ufl.as_vector((fem.Constant(mesh, default_scalar_type(1.0)), fem.Constant(mesh, default_scalar_type(0.2))))
f = fem.Constant(mesh, default_scalar_type(1.0))

a_galerkin = (eps * ufl.dot(ufl.grad(u), ufl.grad(v)) + ufl.dot(b, ufl.grad(u)) * v) * ufl.dx
L_galerkin = f * v * ufl.dx
a_galerkin, L_galerkin


## 2. Add the cellwise SUPG parameter

The learned/optimized quantity is $\tau_K$, represented in a discontinuous, cellwise-constant DG0 space. The residual-weighted streamline term is shown explicitly below; this is the central discretization choice, not hidden behind an API.

In [ ]:
Y = fem.functionspace(mesh, ("DG", 0))
tau = fem.Function(Y, name="tau")
h = ufl.CellDiameter(mesh)
speed = ufl.sqrt(ufl.dot(b, b))
Pe = speed * h / (2 * eps)
tau_expression = h / (2 * speed) * (1 / ufl.tanh(Pe) - 1 / Pe)
tau.interpolate(fem.Expression(tau_expression, Y.element.interpolation_points()))

streamline_test = tau * ufl.dot(b, ufl.grad(v))
strong_residual = -eps * ufl.div(ufl.grad(u)) + ufl.dot(b, ufl.grad(u)) - f
a_supg = a_galerkin + (strong_residual + f) * streamline_test * ufl.dx
L_supg = L_galerkin + f * streamline_test * ufl.dx
tau.x.array.min(), tau.x.array.max()


## 3. State, adjoint, and gradient

For an objective $J(u)$, the package differentiates the residual $R(u,\tau)$ in UFL, solves $R_u^*\psi=J_u$, and assembles $-R_\tau^*\psi$ cellwise. The next cell deliberately exposes those three user-level objects; the reusable solver owns the repeated DOLFINx assembly details.

In [ ]:
from supgml.supg import AdjointSUPGSolver, ConvectionDiffusionProblem

problem = ConvectionDiffusionProblem(mesh, V, u_h, eps, b, None, f, None, [bc])
objective = u_h**2 * ufl.dx
solver = AdjointSUPGSolver(problem, objective)
solver.set_weights(tau.x.array)

print(f"J(tau) = {solver.loss():.3e}")
print(f"number of cellwise parameters = {solver.grad().size}")


## What is intentionally packaged

`AdjointSUPGSolver` owns homogeneous-adjoint boundary conditions, form compilation, sparse solves, local objective assembly, bounds, and SciPy callbacks. Keep new PDE definitions and objectives in notebooks; move only duplicated numerical infrastructure into `src/supgml`. Historical exploratory plots remain in `archive/prototypes`.